In [ ]:
!pip install mediapipe==0.10.14

In [ ]:
# define expressions to detect

num_classes = 2
class_labels = ['happy-face', 'angry-face']


In [ ]:
import cv2

In [ ]:
# import mediapipe library

import mediapipe as mp
mp_facemesh = mp.solutions.face_mesh
mp_draw = mp.solutions.drawing_utils

In [ ]:
# face mesh model
face = mp_facemesh.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True)

In [ ]:
!unzip facial-express.zip

In [ ]:
import os
import numpy as np

x = []
y = []

# build dataset by loading image files for each expression
for expression in class_labels:
  dir_path = os.path.join("facial-express", expression)
  for item in os.listdir(dir_path):
    image_file = os.path.join(dir_path, item)
    if os.path.isfile(image_file):
      image = cv2.imread(image_file)

      if image is not None:

        # convert image to RGB
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # get landmarks
        output = face.process(rgb_image)
        if output.multi_face_landmarks:
          y.append(expression)

          # extract co-ordinates of facial landmarks
          landmarks = []
          for landmark in output.multi_face_landmarks[0].landmark:
            landmarks.append(landmark.x)
            landmarks.append(landmark.y)
            landmarks.append(landmark.z)

          x.append(landmarks)

# x stores landmark points
x = np.array(x)

# y stores corresponding class name
y = np.array(y)

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [ ]:
x.shape

(197, 1434)

In [ ]:
y.shape

(197,)

In [ ]:
# split data for training and testing

from sklearn.model_selection import train_test_split
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size=0.2)

In [ ]:
# choose and run ML model

from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier().fit(train_x, train_y)
model.score(test_x, test_y)

0.875

In [ ]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier().fit(train_x, train_y)
model.score(test_x, test_y)

0.85

In [ ]:
# define neural network architecture

import keras
from keras import layers

inputs = keras.Input(shape=(1434, ))
dense1 = layers.Dense(64, activation="linear")(inputs)
dense2 = layers.Dense(64, activation="linear")(dense1)
dense3 = layers.Dense(64, activation="linear")(dense2)

outputs = layers.Dense(num_classes, activation="softmax")(dense3)
model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 1434)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │        91,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,290 (391.76 KB)

 Trainable params: 100,290 (391.76 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

In [ ]:
# to convert class labels from string to number

from sklearn.preprocessing import LabelEncoder
z = LabelEncoder().fit_transform(y)

In [ ]:
model.fit(x, z, validation_split=0.2, epochs=15, shuffle=True)

Epoch 1/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.8457 - loss: 0.8086 - val_accuracy: 0.0250 - val_loss: 0.9366
Epoch 2/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8648 - loss: 0.5247 - val_accuracy: 0.0000e+00 - val_loss: 3.1164
Epoch 3/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8726 - loss: 0.4011 - val_accuracy: 0.0500 - val_loss: 1.1460
Epoch 4/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8613 - loss: 0.4579 - val_accuracy: 0.0000e+00 - val_loss: 2.0892
Epoch 5/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8487 - loss: 0.4239 - val_accuracy: 0.0000e+00 - val_loss: 2.0724
Epoch 6/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8799 - loss: 0.3440 - val_accuracy: 0.0750 - val_loss: 1.4213
Epoch 7/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8613 - loss: 0.3781 - val_accuracy: 0.0750 - val_loss: 1.6705
Epoch 8/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8899 - loss: 0.3455 - val_accuracy: 0.0500 - val_l